<a href="https://colab.research.google.com/github/juanfloresponce2006-dot/Creaci-n-y-Visualizaci-n-de-KPI-s/blob/main/ProyectoFinal_B2_ISI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install dash pandas plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 60.6 MB/s eta 0:00:00


In [4]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, State, ctx
import time
import random
import csv
import os
from datetime import datetime
from typing import List, Dict

RUTA_CSV = '/content/drive/Othercomputers/Mi portátil/UNIVERSIDAD/TPS/restaurant_orders.csv'


# [FASE 2] BACKEND EN VIVO

class Producto:
    def __init__(self, nombre: str, categoria: str, precio: float):
        self.nombre = nombre
        self.categoria = categoria
        self.precio = precio
        self.stock_teorico = 50

    def descontar_stock(self, cantidad: int):
        self.stock_teorico -= cantidad

class Cliente:
    def __init__(self, nombre: str):
        self.nombre = nombre
        self.compras = 0

class Transaccion:
    def __init__(self, id_orden, cliente, producto, cantidad, precio, fecha, metodo_pago):
        self.id_orden = id_orden
        self.cliente = cliente
        self.producto = producto
        self.cantidad = cantidad
        self.fecha_hora = fecha
        self.monto_total = cantidad * precio
        self.metodo_pago = metodo_pago

class ModuloERP:
    def __init__(self):
        self.caja_chica = 0.0

    def procesar_ingreso(self, tx: Transaccion):
        self.caja_chica += tx.monto_total

class ModuloSCM:
    def __init__(self, umbral_critico: int = 15):
        self.alertas: Dict[str, int] = {}
        self._umbral_critico = umbral_critico

    def procesar_logistica(self, tx: Transaccion):
        tx.producto.descontar_stock(tx.cantidad)
        stock_actual = tx.producto.stock_teorico
        nombre_prod = tx.producto.nombre

        if stock_actual <= self._umbral_critico:
            self.alertas[nombre_prod] = stock_actual

        if stock_actual <= 0:
            tx.producto.stock_teorico += 100
            if nombre_prod in self.alertas:
                del self.alertas[nombre_prod]

class CoreBusinessIntegrado:
    def __init__(self, ruta_csv: str):
        self._ruta = ruta_csv
        self._lineas_leidas = 0
        self.erp = ModuloERP()
        self.scm = ModuloSCM(umbral_critico=15)
        self.clientes = {}
        self.productos = {}
        self.transacciones = []

    def sincronizar(self) -> bool:
        if not os.path.exists(self._ruta): return False
        hubo_cambios = False
        with open(self._ruta, mode='r', encoding='utf-8') as f:
            filas = list(csv.DictReader(f))
            total_actual = len(filas)
            if total_actual > self._lineas_leidas:
                hubo_cambios = True
                for fila in filas[self._lineas_leidas:]:
                    cli_nombre = fila['Customer Name']
                    prod_nombre, cat, precio = fila['Food Item'], fila['Category'], float(fila['Price'])
                    m_pago = fila.get('Payment Method', 'Cash')

                    if cli_nombre not in self.clientes: self.clientes[cli_nombre] = Cliente(cli_nombre)
                    if prod_nombre not in self.productos: self.productos[prod_nombre] = Producto(prod_nombre, cat, precio)

                    try:
                        fecha_orden = datetime.strptime(fila['Order Time'], '%Y-%m-%d %H:%M:%S')
                    except ValueError:
                        continue

                    tx = Transaccion(fila['Order ID'], self.clientes[cli_nombre], self.productos[prod_nombre],
                                     int(fila['Quantity']), precio, fecha_orden, m_pago)

                    self.transacciones.append(tx)
                    self.erp.procesar_ingreso(tx)
                    self.scm.procesar_logistica(tx)

                self._lineas_leidas = total_actual
        return hubo_cambios


# [FASE 3] MOTOR DE INTELIGENCIA DE NEGOCIOS Y APM
class MotorSegmentacionRFM:
    def __init__(self):
        self.df = pd.DataFrame()

    def procesar_desde_core(self, transacciones: List[Transaccion]):
        if not transacciones: return
        datos = [{
            'Order ID': t.id_orden, 'Customer Name': t.cliente.nombre, 'Food Item': t.producto.nombre,
            'Category': t.producto.categoria, 'Quantity': t.cantidad, 'Price': t.producto.precio,
            'Revenue': t.monto_total, 'Order Time': t.fecha_hora, 'Payment Method': t.metodo_pago
        } for t in transacciones]

        self.df = pd.DataFrame(datos)
        trad_cat = {'Main': 'Plato Principal', 'Dessert': 'Postre', 'Starter': 'Entrada', 'Beverage': 'Bebida', 'Side': 'Acompañante'}
        trad_pago = {'Cash': 'Efectivo', 'Credit Card': 'Tarjeta de Crédito', 'Debit Card': 'Tarjeta de Débito', 'Digital Wallet': 'Billetera Digital'}

        self.df['Category'] = self.df['Category'].replace(trad_cat)
        self.df['Payment Method'] = self.df['Payment Method'].replace(trad_pago)

        self.df['Hour'] = self.df['Order Time'].dt.hour
        self.df['Date_str'] = self.df['Order Time'].dt.strftime('%Y-%m-%d')
        self.df['Week_str'] = self.df['Order Time'].dt.strftime('%Y-W%V')
        self.df['Month_str'] = self.df['Order Time'].dt.strftime('%Y-%m')
        self.df['Time_Slot'] = self.df['Hour'].apply(lambda h: 'Mañana' if 6 <= h < 12 else ('Tarde' if 12 <= h < 18 else 'Noche'))

        snapshot = self.df['Order Time'].max()
        rfm = self.df.groupby('Customer Name').agg({
            'Order Time': lambda x: (snapshot - x.max()).days, 'Order ID': 'count', 'Revenue': 'sum'
        }).rename(columns={'Order Time': 'R', 'Order ID': 'F', 'Revenue': 'M'})

        if len(rfm) < 3:
            rfm['R_Score'], rfm['F_Score'], rfm['M_Score'] = 3, 3, 3
        else:
            rfm['R_Score'] = pd.qcut(rfm['R'].rank(method='first'), 3, labels=[3, 2, 1])
            rfm['F_Score'] = rfm['F'].apply(lambda f: 3 if f > 1 else 1)
            rfm['M_Score'] = pd.qcut(rfm['M'].rank(method='first'), 3, labels=[1, 2, 3])

        def seg(row):
            r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
            if f > 1 or (r == 3 and m == 3): return "VIP / Campeones"
            elif r == 3 and m == 2: return "Leales"
            elif r == 2: return "Frecuentes"
            elif r == 1 and m >= 2: return "En Riesgo"
            else: return "Inactivos"

        rfm['Segmento'] = rfm.apply(seg, axis=1)
        self.df = self.df.merge(rfm[['Segmento']], left_on='Customer Name', right_index=True, how='left')

class ModuloCRM:
    def __init__(self):
        self.clientes_riesgo = set()

    def procesar_crm(self, df_completo: pd.DataFrame):
        """Aisla los registros de los clientes y detecta riesgo de deserción"""
        if not df_completo.empty and 'Segmento' in df_completo.columns:
            riesgo = df_completo[df_completo['Segmento'] == 'En Riesgo']['Customer Name'].unique()
            self.clientes_riesgo = set(riesgo)

class ModuloAuditoriaWeb:
    def evaluar_rendimiento(self, tiempo_inicio: float) -> dict:
        t_backend = time.time() - tiempo_inicio
        return {
            "LCP": round(t_backend + random.uniform(0.1, 0.3), 2),
            "INP": round(random.uniform(30.0, 90.0), 1),
            "Estado": "ÓPTIMO" if t_backend < 2.0 else "ALERTA"
        }


# INSTANCIAS GLOBALES
SISTEMA = CoreBusinessIntegrado(RUTA_CSV)
MOTOR_BI = MotorSegmentacionRFM()
APM = ModuloAuditoriaWeb()
CRM = ModuloCRM()


# [FASE 4] FRONT-END Y CONTROLADORES REACTIVOS (DASH)
CARD = {'backgroundColor': 'white', 'borderRadius': '15px', 'padding': '20px', 'boxShadow': '0 4px 15px rgba(0,0,0,0.05)', 'height': '100%'}
H1_ST = {'color': '#0014FF', 'fontSize': '40px', 'margin': '10px 0', 'textAlign': 'center'}
H3_ST = {'fontSize': '18px', 'margin': '0 0 10px 0', 'color': '#333', 'fontWeight': 'bold'}
PLOT_LABELS = {'Hour': 'Hora del Día', 'Revenue': 'Ingresos ($)', 'Quantity': 'Unidades', 'Time_Slot': 'Franja', 'Food Item': 'Plato / Artículo', 'Order Time': 'Fecha', 'Category': 'Categoría', 'Payment Method': 'Método de Pago', 'Week_str': 'Semana', 'Segmento': 'Segmento CRM'}
EMPTY_MSG = html.Div("Esperando transacciones para esta fecha...", style={'textAlign': 'center', 'padding': '50px', 'color': 'gray', 'fontSize': '18px'})

app = Dash(__name__)

app.layout = html.Div(style={'backgroundColor': '#EAE5FF', 'padding': '40px', 'fontFamily': 'Arial, sans-serif', 'minHeight': '100vh'}, children=[
    dcc.Store(id='active-mode', data='M'),
    dcc.Interval(id='auto-refresh', interval=3000, n_intervals=0),

    html.Div(style={'maxWidth': '1200px', 'margin': '0 auto'}, children=[
        html.Div(id='panel-tecnico', style={'backgroundColor': '#2C1B4D', 'color': 'white', 'padding': '10px 20px', 'borderRadius': '10px', 'marginBottom': '20px', 'display': 'flex', 'justifyContent': 'space-between', 'fontSize': '14px', 'fontWeight': 'bold'}),

        # CABECERA Y CONTROLES GLOBALES
        html.Div(style={'display': 'flex', 'justifyContent': 'space-between', 'marginBottom': '10px', 'alignItems': 'center'}, children=[
            html.H1("Tablero Corporativo", style={'margin': '0', 'color': '#2C1B4D'}),
            html.Div(style={'display': 'flex', 'gap': '15px', 'alignItems': 'center'}, children=[
                dcc.Dropdown(id='time-selector', clearable=False, style={'width': '220px', 'borderRadius': '5px'}),
                html.Div(style={'backgroundColor': '#E0DBEC', 'borderRadius': '30px', 'padding': '8px 12px', 'display': 'flex', 'gap': '10px'}, children=[
                    html.Button('D', id='btn-d', n_clicks=0, style={'border': 'none', 'borderRadius': '50%', 'width': '50px', 'height': '50px', 'fontSize': '20px', 'fontWeight': 'bold', 'cursor': 'pointer', 'transition': '0.3s'}),
                    html.Button('W', id='btn-w', n_clicks=0, style={'border': 'none', 'borderRadius': '50%', 'width': '50px', 'height': '50px', 'fontSize': '20px', 'fontWeight': 'bold', 'cursor': 'pointer', 'transition': '0.3s'}),
                    html.Button('M', id='btn-m', n_clicks=0, style={'border': 'none', 'borderRadius': '50%', 'width': '50px', 'height': '50px', 'fontSize': '20px', 'fontWeight': 'bold', 'cursor': 'pointer', 'transition': '0.3s'})
                ])
            ])
        ]),

        # SISTEMA DE PESTAÑAS (SPA)
        dcc.Tabs(id='tabs-selector', value='tab-bi', children=[
            dcc.Tab(label='📡 Inteligencia de Negocios (BI)', value='tab-bi', style={'fontWeight': 'bold'}, selected_style={'backgroundColor': '#2C1B4D', 'color': 'white', 'fontWeight': 'bold'}),
            dcc.Tab(label='💰 Finanzas (ERP)', value='tab-erp', style={'fontWeight': 'bold'}, selected_style={'backgroundColor': '#2C1B4D', 'color': 'white', 'fontWeight': 'bold'}),
            dcc.Tab(label='📦 Logística (SCM)', value='tab-scm', style={'fontWeight': 'bold'}, selected_style={'backgroundColor': '#2C1B4D', 'color': 'white', 'fontWeight': 'bold'})
        ], style={'marginBottom': '20px'}),

        html.Div(id='dashboard-content')
    ])
])

@app.callback(
    [Output('time-selector', 'options'), Output('time-selector', 'value'),
     Output('btn-d', 'style'), Output('btn-w', 'style'), Output('btn-m', 'style'), Output('active-mode', 'data'),
     Output('panel-tecnico', 'children'), Output('dashboard-content', 'children')],
    [Input('tabs-selector', 'value'), Input('btn-d', 'n_clicks'), Input('btn-w', 'n_clicks'), Input('btn-m', 'n_clicks'), Input('auto-refresh', 'n_intervals')],
    [State('time-selector', 'value'), State('active-mode', 'data')]
)
def orquestador_maestro(tab_val, d_clicks, w_clicks, m_clicks, n_intervals, current_val, current_mode):
    t_inicio = time.time()

    # LECTURA DEL BACKEND
    hubo_cambios = SISTEMA.sincronizar()
    if hubo_cambios or MOTOR_BI.df.empty:
        MOTOR_BI.procesar_desde_core(SISTEMA.transacciones)
    df = MOTOR_BI.df

    # EJECUTAR CRM
    CRM.procesar_crm(df)

    # PANEL TÉCNICO
    apm_stats = APM.evaluar_rendimiento(t_inicio)
    alertas_scm = " | ".join([f"⚠ {p} ({s})" for p, s in SISTEMA.scm.alertas.items()]) if SISTEMA.scm.alertas else "Stock Estable"
    alertas_crm = f"⚠ {len(CRM.clientes_riesgo)} Clientes en Riesgo de Deserción" if CRM.clientes_riesgo else "CRM Óptimo"

    panel_tech = [
        html.Span(f"📜 ERP (Caja Chica): ${SISTEMA.erp.caja_chica:,.2f}", style={'color': '#39FF14'}),
        html.Span(f"📦 SCM (Inventario): {alertas_scm}", style={'color': '#FF3939' if SISTEMA.scm.alertas else '#00FFFF'}),
        html.Span(f"🤝 CRM (Retención): {alertas_crm}", style={'color': '#FF3939' if CRM.clientes_riesgo else '#39FF14'}),
        html.Span(f"⚡ APM: LCP {apm_stats['LCP']}s | INP {apm_stats['INP']}ms")
    ]

    def_s = {'border': 'none', 'borderRadius': '50%', 'width': '50px', 'height': '50px', 'fontSize': '20px', 'fontWeight': 'bold', 'backgroundColor': '#E0DBEC', 'color': '#2C1B4D'}
    act_s = def_s.copy(); act_s['backgroundColor'] = '#A899CC'

    if df.empty: return [], None, def_s, def_s, def_s, 'M', panel_tech, html.Div("Ingresa una transacción...", style={'textAlign': 'center', 'padding': '50px'})

    # LÓGICA DE TIEMPO
    trigger = ctx.triggered_id
    mode = trigger.replace('btn-', '').upper() if trigger in ['btn-d', 'btn-w', 'btn-m'] else current_mode
    sd, sw, sm = def_s.copy(), def_s.copy(), def_s.copy()

    if mode == 'D':
        opts = [{'label': d, 'value': d} for d in df['Date_str'].sort_values(ascending=False).unique()]
        val = current_val if current_val in [o['value'] for o in opts] and trigger not in ['btn-d', 'btn-w', 'btn-m'] else opts[0]['value']
        sd = act_s
        df_curr = df[df['Date_str'] == val]
        df_prev = pd.DataFrame() # No WoW/MoM for daily
    elif mode == 'W':
        opts = [{'label': f"Semana {w.split('-W')[1]} ({w.split('-W')[0]})", 'value': w} for w in df['Week_str'].sort_values(ascending=False).unique()]
        val = current_val if current_val in [o['value'] for o in opts] and trigger not in ['btn-d', 'btn-w', 'btn-m'] else opts[0]['value']
        sw = act_s
        df_curr = df[df['Week_str'] == val]
        max_date = df_curr['Order Time'].max() if not df_curr.empty else datetime.now()
        df_prev = df[(df['Order Time'] >= max_date - pd.Timedelta(days=14)) & (df['Order Time'] < max_date - pd.Timedelta(days=7))]
    else:
        opts = [{'label': m, 'value': m} for m in df['Month_str'].sort_values(ascending=False).unique()]
        val = current_val if current_val in [o['value'] for o in opts] and trigger not in ['btn-d', 'btn-w', 'btn-m'] else opts[0]['value']
        sm = act_s
        df_curr = df[df['Month_str'] == val]
        max_date = df_curr['Order Time'].max() if not df_curr.empty else datetime.now()
        df_prev = df[(df['Order Time'] >= max_date - pd.Timedelta(days=60)) & (df['Order Time'] < max_date - pd.Timedelta(days=30))]

    # ENRUTADOR DE PESTAÑAS
    if tab_val == 'tab-erp': layout = build_layout_erp(mode, df_curr, df_prev)
    elif tab_val == 'tab-scm': layout = build_layout_scm(mode, df_curr, df_prev)
    else: layout = build_layout_bi(mode, df_curr, df_prev)

    return opts, val, sd, sw, sm, mode, panel_tech, layout

# 1: BI (Inteligencia de Negocios)
def build_layout_bi(mode, df_curr, df_prev):
    if df_curr.empty: return EMPTY_MSG

    if mode == 'D':
        fig1 = px.bar(df_curr.groupby('Hour')['Revenue'].sum().reset_index(), x='Hour', y='Revenue', labels=PLOT_LABELS, template='plotly_white', color_discrete_sequence=['#1f4899'])
        fig2 = px.pie(df_curr, names='Category', values='Quantity', labels=PLOT_LABELS, hole=0.5)
        fig2.update_traces(textposition='inside', textinfo='percent+label'); fig2.update_layout(margin=dict(t=10, b=10, l=10, r=10), showlegend=False)
        fig3 = px.line(df_curr.groupby('Time_Slot')['Revenue'].sum().reset_index(), x='Time_Slot', y='Revenue', labels=PLOT_LABELS, markers=True, template='plotly_white', color_discrete_sequence=['#39FF14'])
        item_qty = df_curr.groupby('Food Item')['Quantity'].sum().sort_values()
        top_bottom = pd.concat([item_qty.head(3), item_qty.tail(3)]).reset_index() if len(item_qty) >= 6 else item_qty.reset_index()
        fig4 = px.bar(top_bottom, y='Food Item', x='Quantity', labels=PLOT_LABELS, orientation='h', template='plotly_white', color_discrete_sequence=['#0b2d71'])

        return html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '20px'}, children=[
            html.Div(style=CARD, children=[html.H3("Ingresos por Horas", style=H3_ST), dcc.Graph(figure=fig1, style={'height': '240px'})]),
            html.Div(style=CARD, children=[html.H3("Composición del Ticket", style=H3_ST), dcc.Graph(figure=fig2, style={'height': '240px'})]),
            html.Div(style=CARD, children=[html.H3("Ventas por Franja Horaria", style=H3_ST), dcc.Graph(figure=fig3, style={'height': '240px'})]),
            html.Div(style=CARD, children=[html.H3("Top vs Bottom Platos", style=H3_ST), dcc.Graph(figure=fig4, style={'height': '240px'})])
        ])

    elif mode == 'W':
        rev_w = df_curr['Revenue'].sum(); rev_prev_w = df_prev['Revenue'].sum() if not df_prev.empty else 0
        wow = ((rev_w - rev_prev_w) / rev_prev_w * 100) if rev_prev_w else 0
        fig1 = px.bar(df_curr.groupby('Food Item')['Quantity'].sum().sort_values().reset_index(), x='Food Item', y='Quantity', labels=PLOT_LABELS, template='plotly_white', color_discrete_sequence=['#A899CC'])
        w_cust = df_curr.groupby('Customer Name')['Order ID'].nunique()

        return html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '20px'}, children=[
            html.Div(style=CARD, children=[html.H3("Ventas Semanales", style=H3_ST), html.H1(f"${rev_w:,.2f}", style=H1_ST), html.H3(f"{'🚀 +' if wow >= 0 else '🔻 '}{wow:.2f}% WoW", style={'color': '#39FF14' if wow >= 0 else 'red', 'textAlign': 'center'})]),
            html.Div(style=CARD, children=[html.H3("Ticket Promedio", style=H3_ST), html.H1(f"${rev_w / df_curr['Order ID'].nunique():,.2f}", style=H1_ST)]),
            html.Div(style=CARD, children=[html.H3("Desempeño del Menú", style=H3_ST), dcc.Graph(figure=fig1, style={'height': '240px'})]),
            html.Div(style=CARD, children=[html.H3("Recurrencia (CRM)", style=H3_ST), html.H1(f"{(len(w_cust[w_cust > 1]) / len(w_cust) * 100) if len(w_cust) else 0:.1f}%", style=H1_ST), html.P("Clientes con >1 compra.", style={'textAlign': 'center', 'color': 'gray'})])
        ])

    else:
        rev_m = df_curr['Revenue'].sum(); rev_prev_m = df_prev['Revenue'].sum() if not df_prev.empty else 0
        mom = ((rev_m - rev_prev_m) / rev_prev_m * 100) if rev_prev_m else 0
        fig_area = px.area(df_curr.groupby(df_curr['Order Time'].dt.date)['Revenue'].sum().reset_index(), x='Order Time', y='Revenue', labels=PLOT_LABELS, template='plotly_white')
        fig_area.update_traces(line_color='#0b2d71', fillcolor='rgba(11, 45, 113, 0.2)'); fig_area.update_layout(margin=dict(l=0,r=0,t=10,b=0), height=150)
        fig_rfm = px.pie(names=df_curr['Segmento'].value_counts().index, values=df_curr['Segmento'].value_counts().values, hole=0.6, color_discrete_sequence=['#0b2d71', '#1f4899', '#3a66c4', '#6086d6', '#8ea9e5'])
        fig_rfm.update_traces(textposition='outside', textinfo='label+percent'); fig_rfm.update_layout(showlegend=False, margin=dict(t=10, b=10, l=10, r=10))
        fig_tree = px.treemap(df_curr, path=['Payment Method'], values='Revenue', labels=PLOT_LABELS, color='Payment Method', color_discrete_sequence=px.colors.qualitative.Pastel)
        fig_tree.update_traces(textinfo="label+percent entry+value", textfont_size=14); fig_tree.update_layout(margin=dict(t=10, b=10, l=10, r=10))
        fig_cat = px.bar(df_curr.groupby(['Week_str', 'Category'])['Revenue'].sum().reset_index(), x='Week_str', y='Revenue', color='Category', labels=PLOT_LABELS, barmode='relative', template='plotly_white')

        # SECCIÓN CRM VISUAL
        riesgo_html = [html.Div(f"⚠ Aislar registro: {c} - Iniciar campaña de recuperación", style={'backgroundColor': '#ffcccc', 'color': 'red', 'padding': '8px', 'marginBottom': '5px', 'borderRadius': '5px', 'fontWeight': 'bold'}) for c in CRM.clientes_riesgo]
        if not riesgo_html: riesgo_html = [html.Div("No hay clientes en riesgo crítico.", style={'color': 'green'})]

        card_crm = html.Div(style={**CARD, 'gridColumn': '1 / -1'}, children=[
            html.H3("🚨 Alertas CRM: Clientes en Riesgo de Deserción", style=H3_ST),
            html.Div(riesgo_html, style={'height': '120px', 'overflowY': 'auto', 'padding': '10px'})
        ])

        return html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '20px'}, children=[
            html.Div(style=CARD, children=[html.H3("Ingresos Consolidados", style=H3_ST), html.H1(f"${rev_m:,.2f}", style=H1_ST), html.H4(f"{'📈 +' if mom >= 0 else '📉 '}{mom:.2f}% MoM", style={'color': '#39FF14' if mom >= 0 else 'red', 'textAlign': 'center'}), dcc.Graph(figure=fig_area)]),
            html.Div(style=CARD, children=[html.H3("Segmentación RFM", style=H3_ST), dcc.Graph(figure=fig_rfm, style={'height': '240px'})]),
            html.Div(style=CARD, children=[html.H3("Concentración de Pagos", style=H3_ST), dcc.Graph(figure=fig_tree, style={'height': '260px'})]),
            html.Div(style=CARD, children=[html.H3("Categorías", style=H3_ST), dcc.Graph(figure=fig_cat, style={'height': '260px'})]),
            card_crm # Insertamos el CRM al final
        ])

# FÁBRICA 2: ERP (Finanzas)
def build_layout_erp(mode, df_curr, df_prev):
    if df_curr.empty: return EMPTY_MSG

    ingresos_periodo = df_curr['Revenue'].sum()
    transacciones_totales = df_curr['Order ID'].nunique()
    ticket_promedio = ingresos_periodo / transacciones_totales if transacciones_totales else 0

    if mode == 'D':
        fig_evol = px.line(df_curr.groupby('Hour')['Revenue'].sum().reset_index(), x='Hour', y='Revenue', labels=PLOT_LABELS, markers=True, template='plotly_white', color_discrete_sequence=['#2C1B4D'])
    else:
        fig_evol = px.area(df_curr.groupby(df_curr['Order Time'].dt.date)['Revenue'].sum().reset_index(), x='Order Time', y='Revenue', labels=PLOT_LABELS, template='plotly_white')
        fig_evol.update_traces(line_color='#2C1B4D', fillcolor='rgba(44, 27, 77, 0.2)')

    fig_evol.update_layout(margin=dict(l=0,r=0,t=10,b=0))

    fig_pagos = px.pie(df_curr, names='Payment Method', values='Revenue', labels=PLOT_LABELS, hole=0.4, color_discrete_sequence=px.colors.sequential.Teal)
    fig_pagos.update_traces(textposition='inside', textinfo='percent+label'); fig_pagos.update_layout(margin=dict(t=10, b=10, l=10, r=10), showlegend=False)

    return html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '20px'}, children=[
        html.Div(style=CARD, children=[
            html.H3("Estado Contable (RAM)", style=H3_ST),
            html.H1(f"${SISTEMA.erp.caja_chica:,.2f}", style={'color': '#27ae60', 'fontSize': '48px', 'margin': '10px 0', 'textAlign': 'center'}),
            html.P("Caja Chica Global Acumulada en Tiempo Real", style={'textAlign': 'center', 'color': 'gray'})
        ]),
        html.Div(style=CARD, children=[
            html.H3(f"Ingresos del Periodo ({mode})", style=H3_ST),
            html.H1(f"${ingresos_periodo:,.2f}", style=H1_ST),
            html.H4(f"Ticket Promedio: ${ticket_promedio:,.2f} | Transacciones: {transacciones_totales}", style={'textAlign': 'center', 'color': 'gray'})
        ]),
        html.Div(style=CARD, children=[html.H3("Evolución de Ingresos", style=H3_ST), dcc.Graph(figure=fig_evol, style={'height': '260px'})]),
        html.Div(style=CARD, children=[html.H3("Distribución de Pagos", style=H3_ST), dcc.Graph(figure=fig_pagos, style={'height': '260px'})])
    ])

# FÁBRICA 3: SCM (Logística)
def build_layout_scm(mode, df_curr, df_prev):
    if df_curr.empty: return EMPTY_MSG

    unidades_vendidas = df_curr['Quantity'].sum()

    alertas_html = []
    if not SISTEMA.scm.alertas:
        alertas_html = [html.Div("✔️ Inventario Sano. No hay alertas críticas.", style={'color': 'green', 'textAlign': 'center', 'padding': '20px', 'fontWeight': 'bold'})]
    else:
        for prod, stock in SISTEMA.scm.alertas.items():
            alertas_html.append(html.Div(f"⚠ URGENTE: Reabastecer {prod} (Quedan {stock} unidades)", style={'backgroundColor': '#ffeaa7', 'color': '#d35400', 'padding': '10px', 'borderRadius': '5px', 'marginBottom': '10px', 'fontWeight': 'bold'}))

    card_rotacion = html.Div(style=CARD, children=[
        html.H3(f"Rotación de Inventario ({mode})", style=H3_ST),
        html.H1(f"{unidades_vendidas:,} unds", style=H1_ST),
        html.P("Total de artículos despachados en el periodo.", style={'textAlign': 'center', 'color': 'gray'})
    ])

    card_alertas = html.Div(style=CARD, children=[
        html.H3("Panel de Alertas SCM (Tiempo Real)", style=H3_ST),
        html.Div(alertas_html, style={'height': '220px', 'overflowY': 'auto'})
    ])

    if mode == 'D':
        return html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '20px'}, children=[
            card_rotacion, card_alertas
        ])
    else:
        agrupado = df_curr.groupby('Food Item')['Quantity'].sum().sort_values(ascending=False).reset_index()
        fig_top = px.bar(agrupado.head(5), x='Food Item', y='Quantity', labels=PLOT_LABELS, template='plotly_white', color_discrete_sequence=['#27ae60'])
        fig_bot = px.bar(agrupado.tail(5), x='Food Item', y='Quantity', labels=PLOT_LABELS, template='plotly_white', color_discrete_sequence=['#e74c3c'])

        card_top = html.Div(style=CARD, children=[html.H3("Top 5 Alta Rotación", style=H3_ST), dcc.Graph(figure=fig_top, style={'height': '260px'})])
        card_bot = html.Div(style=CARD, children=[html.H3("Top 5 Baja Rotación", style=H3_ST), dcc.Graph(figure=fig_bot, style={'height': '260px'})])

        return html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '20px'}, children=[
            card_rotacion, card_alertas, card_top, card_bot
        ])

if __name__ == '__main__':
    app.run(jupyter_mode='external')

Dash app running on:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>